In [1]:
import os
from dotenv import load_dotenv
from google import genai

# Carga la API key guardada en el archivo .env de la raíz
load_dotenv()

# Inicializa el cliente oficial del SDK google-genai
client = genai.Client()

# Llamada de prueba al modelo
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="¡Hola desde un Jupyter Notebook! Confirma que la configuración funciona correctamente.",
)

print("Respuesta de Gemini:\n")
print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Respuesta de Gemini:

¡Hola! 👋

**¡Configuración confirmada!** La conexión desde tu Jupyter Notebook está funcionando perfectamente.

Estoy listo para ayudarte. Desde aquí podemos trabajar en:

* 🐍 **Código en Python** (Pandas, NumPy, Matplotlib, Scikit-learn, etc.).
* 📊 **Análisis y visualización de datos**.
* 🐛 **Depuración de errores** (debugging) en tus celdas.
* 📝 **Explicación de conceptos** o generación de documentación en Markdown.

¿En qué proyecto o tarea te gustaría empezar a trabajar hoy?


In [2]:
prompt_poblacion = (
    "¿Cuál es la población actual de Mallorca? "
    "Dame la cifra total aproximada y un breve desglose por sus municipios principales."
)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt_poblacion,
)

print(response.text)

La población residente actual de la isla de Mallorca es de aproximadamente **950.000 habitantes** (según los últimos datos oficiales del Instituto Nacional de Estadística - INE, correspondientes a 2023/2024). 

Mallorca concentra cerca del 80% de toda la población del archipiélago balear.

A continuación, te presento un desglose aproximado de sus **municipios principales** ordenados por número de habitantes:

1. **Palma (la capital):** **~425.000 habitantes.** Concentra casi el 45% de toda la población de la isla. Es el centro político, económico y cultural.
2. **Calvià:** **~53.000 habitantes.** Ubicado en el suroeste, es uno de los motores turísticos de la isla y cuenta con una gran comunidad de residentes extranjeros.
3. **Manacor:** **~46.000 habitantes.** Es la capital de la comarca del Llevant (el este de la isla) y la segunda ciudad en importancia comercial e industrial.
4. **Marratxí:** **~39.000 habitantes.** Municipio colindante con Palma que ha experimentado un gran crecimie

In [3]:
from google.genai import types

def get_completion(prompt: str, model: str = "gemini-3.6-flash", temperature: float = 0.0) -> str:
    config = types.GenerateContentConfig(
        temperature=temperature
    )
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=config
    )
    return response.text

# Test básico
print(get_completion("What is 1+1?"))

1 + 1 = 2


In [4]:
customer_email = """
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls 
with smoothie! And to make matters worse, the warranty don't cover the cost of 
cleaning up me kitchen. I need yer help right now, matey!
"""

style = "American English in a calm and respectful tone"

prompt = f"""Translate the text that is delimited by triple backticks \
into a style that is {style}.
text: ```{customer_email}```
"""

response_text = get_completion(prompt)
print("Traducción a estilo calmado:\n", response_text)

Traducción a estilo calmado:
 I am very frustrated that my blender lid came off and splattered smoothie all over my kitchen walls. To make matters worse, it seems the warranty does not cover the cost of cleaning up the mess. I would really appreciate your assistance with this as soon as possible.


In [5]:
from pydantic import BaseModel, Field

# 1. Definimos el esquema de datos tipado
class ReviewExtraction(BaseModel):
    gift: bool = Field(
        description="Was the item purchased as a gift for someone else? True if yes, False if not or unknown."
    )
    delivery_days: int = Field(
        description="How many days did it take for the product to arrive? If not found, output -1."
    )
    price_value: list[str] = Field(
        description="Sentences about the value or price as a list of strings."
    )

customer_review = """
This leaf blower is pretty amazing. It has four settings: candle blower, 
gentle breeze, windy city, and tornado. It arrived in two days, just in time 
for my wife's anniversary present. I think my wife liked it so much she was speechless. 
So far I've been the only one using it, and I've been using it every other morning 
to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers 
out there, but I think it's worth it for the extra features.
"""

# 2. Configuración con response_schema nativo (JSON validado)
config_estructurada = types.GenerateContentConfig(
    temperature=0.0,
    response_mime_type="application/json",
    response_schema=ReviewExtraction,
)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=f"Extract the requested fields from this review:\n\n{customer_review}",
    config=config_estructurada
)

# 3. Acceso directo a los datos
print("Respuesta JSON bruta:")
print(response.text)

# Si tienes Pydantic instalado, puedes parsearlo a un objeto directamente:
datos_extraidos = ReviewExtraction.model_validate_json(response.text)
print("\nAcceso a campo tipado:")
print(f"¿Fue un regalo?: {datos_extraidos.gift}")
print(f"Días de entrega: {datos_extraidos.delivery_days}")
print(f"Frases de precio: {datos_extraidos.price_value}")

Respuesta JSON bruta:
{"gift":true,"delivery_days":2,"price_value":["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]}

Acceso a campo tipado:
¿Fue un regalo?: True
Días de entrega: 2
Frases de precio: ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]


In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# 1. Initialize the model using LangChain with the exact model identifier
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.0,
)

# 2. Define a clean, reusable prompt template using LCEL conventions
translation_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert communications assistant. Adapt the style while preserving the exact core meaning."),
    ("human", "Translate the text delimited by triple backticks into a style that is {style}.\n\ntext: ```{text}```")
])

# 3. Create the chain using the LCEL pipe operator (|)
translation_chain = translation_prompt | llm

# 4. Invoke the chain
customer_email = """
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls 
with smoothie! And to make matters worse, the warranty don't cover the cost of 
cleaning up me kitchen. I need yer help right now, matey!
"""

response = translation_chain.invoke({
    "style": "American English in a calm and respectful tone",
    "text": customer_email
})

print("--- LangChain Response ---")
print(response.content)

c:\Users\ToniF\miniconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- LangChain Response ---
[{'type': 'text', 'text': 'I am very frustrated that my blender lid came off, splattering smoothie all over my kitchen walls. To make matters worse, the warranty does not cover the cost of cleaning up the mess. I would greatly appreciate your prompt assistance with this matter.', 'extras': {'signature': 'EqoZCqcZAWkUfRPAo+uFFIY/xCR+hyitveBinSAiG8UYbMNYsTqVn6lUaDCEKejjIRarBn53wfDJcV72hFpy75xPnhZ2R2NvLmMEz84n5AS3J5w0AEJ1txCjlMnJicQfjbUGJGs8rUyNjFRz7UG+54STXKYIresoH4rw/y4U4WyEgfpBampwREfM7R1EtVUsPP2SBAj/RaDsIAMiNNFqd6/5IsFiDnOVGSKqoSAlj82P7IpmDFkdUu2Qjt0fjRQvpnrzLhr2WgNeqK1dVT5Qu1/R/BBTUpeq71KOh8CKV6Qc5JfIP8/leNHMixB0B3iJFInvygxiIv60vApoQPWZuanEha0/SzoDrZPYOJ/R5ulYWdD/hEZE3GxwWq1pQWNI6dimXOV7zkh1W2ZCJmFzFYVELm9o5JBwiTW7xqM8q9dY8/hNAzT3+ecws5/yNfVB4sHs1sjgpUdosBbod0XoBqsxzVqYChgHd5e074nyMgfXoQ86A1gxFpv2PelagkI7QqIF0E4eHQ/JlLhaHgQhTedrzL0lsQzT8H5OTTA4+cQ7kn6wtIkhA9jiKBXziTfTac53uSMavi+yZS4I454RhHuCKqyhQQJuZACsTc49eOJjnLdldBS1QrehM6PkJPYFSMMNL8WRwwxK8vcAapMrrt/GHr3

In [8]:
from pydantic import BaseModel, Field

# 1. Define the target schema
class IncidentReport(BaseModel):
    summary: str = Field(description="A concise one-sentence summary of the core incident.")
    department: str = Field(description="Responsible department: Logistics, Operations, Customer Support, or Billing.")
    urgency_level: str = Field(description="Urgency classification: Low, Medium, or Critical.")
    action_items: list[str] = Field(description="List of concrete, actionable next steps required.")

# 2. Bind the schema directly to the model
structured_llm = llm.with_structured_output(IncidentReport)

# 3. Build the extraction chain
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract structured information from the provided raw input text."),
    ("human", "{raw_input}")
])

extraction_chain = extraction_prompt | structured_llm

# 4. Run inference
raw_audio_transcript = (
    "Hey team, quick update: the barcode scanner network in the central warehouse just went down "
    "and we have three trucks waiting to load. We need to restart the local server immediately "
    "and notify on-call infrastructure support because shipments are completely blocked."
)

report: IncidentReport = extraction_chain.invoke({"raw_input": raw_audio_transcript})

print("--- Structured Object Extracted ---")
print(f"Summary:     {report.summary}")
print(f"Department:  {report.department}")
print(f"Urgency:     {report.urgency_level}")
print(f"Action Items: {report.action_items}")

c:\Users\ToniF\miniconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- Structured Object Extracted ---
Summary:     The central warehouse barcode scanner network is down, halting all shipment loading.
Department:  Logistics
Urgency:     Critical
Action Items: ['Restart the local server immediately', 'Notify on-call infrastructure support']


In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. Initialize Gemini with temperature=0.0
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.0,
)

# 2. Define prompt with dynamic history placeholder
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful and concise assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

base_chain = chat_prompt | llm

# 3. Session storage dictionary (in production, use Redis or PostgreSQL)
session_store = {}

def get_session_history(session_id: str):
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

# 4. Wrap the LCEL chain with history management
conversational_chain = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 5. Multi-turn test execution
config = {"configurable": {"session_id": "session_001"}}

res1 = conversational_chain.invoke({"input": "Hi, my name is Andrew."}, config=config)
print("Turn 1:", res1.content)

res2 = conversational_chain.invoke({"input": "What is 1 + 1?"}, config=config)
print("Turn 2:", res2.content)

res3 = conversational_chain.invoke({"input": "What is my name?"}, config=config)
print("Turn 3:", res3.content)

C:\Users\ToniF\AppData\Local\Temp\ipykernel_58568\2144716536.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
c:\Users\ToniF\miniconda3\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
C:\Users\ToniF\AppData\Local\Temp\ipykernel_58568\2144716536.py:26: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  session_store[session_id] = Ch

Turn 1: [{'type': 'text', 'text': 'Hello Andrew! Nice to meet you. How can I help you today?', 'extras': {'signature': 'EqUGCqIGAWkUfRMxHsh7INZQMOSh/W8oqy8vV1XXkm8hvbAe+Xq2G0WXz8o+QPl4GVHguCquhP5ohVKmTb7WsRwt7StwuNL5VkdEcuWfTwq+FR2P2IcAiKL7t68Su5ZMXYXB0UHfPLZwKPUIJy2jB0OdNdmFSj9QFnqYmujvLG/sgWvOnE5oZ0Tw8aWHYD+Ys03rlV2knNSN4LF5jyyrCovjfuQy3UyirmXYjlAzJs0d1AZ3kR28OTRbShTZUaBLkSzNLqRhHMfUxhoBO1jh2fnpR7Dha6lns12Z1XWi0nn+9hK6ajWL9DFLvKj+HL/zcVqqnD2ProGCajHlBhEesgE61kdisx3t98f5KyBjsy6aKwCkf4IHAWg5EQKfGAiqJCbbHdd4R7VQrYE1ba5ZFbwHoNn79Xv6xLZL3CXKmRbxQrOUdH5LkMpgAUoZSIcXTeDYLKoJbehEZh3LlSJBzBsSTgIgW8f84hWnDYVj+oIW7+RpZWmJNIkvqrFSqDoirZ5JyT1UIzbnpts8EHPn0DIaq+W73mWqrhbb5NiyxMpN3P9y9+QNeQF+GXUzKq1mF0wBNIqnF8H+khwFlh/08f4dmNXGDzxyO5DE0zI/G+knKV9lJa8Lpt/saqng7cTON6QqZe8pQDjMRsU91idpVu8wR9aLUKK7VK36DfnNpVggmfLtCaTFxo9XpJag9WR1+M3qmfj1ttzmO+6DT9pQTbIPyWKmCYxEpY3rvhTmZBl+a+b+TlOJGZGThhDqOWaKg4vQHEOxPqlLZ11LAvTeZUqwniJWXaZs+NB3T+neX8BwVal/Xh3mc2lRPqgXat6tYmwOF3WbPWdieJtUvm4d9QEFb8OSnVvfoD1dDwPaY+vMvVmo

c:\Users\ToniF\miniconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Turn 2: [{'type': 'text', 'text': '1 + 1 = 2.', 'extras': {'signature': 'Er0DCroDAWkUfRP0cg3TRtlhX0Ucwh0O0xdZFZw0UXVl+TfzmQNC56yS2Tc8FhIGHLyssp39uuHeFvyBdQmuo1S8C61NhSI3cb5IMw78oxXrpr03v5nJsw6GOFOc2RejmZyDKc5Usa//6JsMMfJrutlsAGwlSwP40vSFXM2gKY1LnKhc2VVu2Ii3H2ybuvzt3x6DdhUJ81AH5S1K0lMvtv6MztwZ3Mmf6uxY7HBvcfrAgbtLXWNhjnSbpCcXRaFhA0QA1l/CY3NUMEG7ElrxTo8UoM0fPm6BB+3a1osB53ZbavUCwX/uf8/XjPeGRHnpulhgCYuW8FBCmi5L9EIepRdDyjMi3Q7eMgcXfZtYNV8CLsr/iiXaOSlzIOMat484TQGtQ4+RgvkaXfWrmjiITSvG6fxiDTxiXJfD5RCFQMMNcvdQVxGLZDfwceP+sVpMqaiTSW9yqO76ukLd+Sd7GX2wzWwmy6vqfd+80YnHRAmTrrmThhtLLPUsIuMxDmou/JA/gUeUqmf9Rw/Gc7+vj/oJ7uerwXQB6nAk5psfX72McXMYtNtzFD3kGCVaNtfeg92T4FcXhYyjEG5kB0/RWg=='}}]


c:\Users\ToniF\miniconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Turn 3: [{'type': 'text', 'text': 'Your name is Andrew.', 'extras': {'signature': 'Ep0DCpoDAWkUfRN6E7xd9ViuZS8T5WamsrQ97E5TmQqGvvikY9WAJR2VOOc3eQRsai8Dv5EPntJ3AkA4hp5J1uxcpyLIBgddkvJHNr4eHUmUc+Cb5HDQoPg2llI2eN56XRDFx/UfAvAKeM8WMRVmDc6yMGf6RurvakRjjLZWvjDJGyLLGXfFNve1fUBI6PXl/DsqIsbEokCcnNOgr0OtNDeFunOE/7ER16/tU81m9m3NxZ6k4kvoai1q2S9sucEroGbAvh3FESW5D8OA4qd/GJm86A9GmHyFqUDuKmZ7wiKc+GsaiJoa5UGStV3vXtTzcxRKiES9EjbUZGdisZ3epZkzu83PqBVh1UKEzbnVjAa4O5YAsv89iYgYBbu5K50CMybJTy8OHiqt5wSqjp8P9Vzl+LP27bVytXZjB1HbqFUNs5SBFp8UzT3/mqYPXJUkDDZj1HSxMFOpHFV4jrMUEKzeqL+XIKY3GD74LCP/UfdQmstEjHIMocHnZOZgskl6N803Fo/H4lZ0jeHvfIkyHzCHUeGPg7/8vjNGPuqK9Vg='}}]


In [11]:
from langchain_core.messages import HumanMessage, AIMessage, trim_messages

# 1. Simulated dialogue history
raw_history = [
    HumanMessage(content="Hi, my name is Andrew."),
    AIMessage(content="Hello Andrew, nice to meet you!"),
    HumanMessage(content="What is 1 + 1?"),
    AIMessage(content="1 + 1 equals 2."),
    HumanMessage(content="AI is what?"),
    AIMessage(content="AI is amazing!"),
    HumanMessage(content="Backpropagation is what?"),
    AIMessage(content="Backpropagation is beautiful!")
]

# 2. Prune history to retain only the last 4 messages (2 exchanges)
pruned_history = trim_messages(
    raw_history,
    strategy="last",
    token_counter=len,  # Counts message turns
    max_tokens=4,
    start_on="human",
    include_system=True
)

print("--- Trimmed Context Passed to Model ---")
for msg in pruned_history:
    print(f"[{msg.type}]: {msg.content}")

--- Trimmed Context Passed to Model ---
[human]: AI is what?
[ai]: AI is amazing!
[human]: Backpropagation is what?
[ai]: Backpropagation is beautiful!


In [12]:
summarizer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Condense the provided schedule/meeting text into a concise, bulleted executive summary."),
    ("human", "{raw_schedule}")
])

summarizer_chain = summarizer_prompt | llm

schedule_text = """
There is a meeting at 8:00 AM with the product team regarding the Q3 roadmap.
From 9:00 AM to 12:00 PM, dedicate time to work on LangChain data pipelines.
At noon, lunch at the Italian restaurant with a customer who wants to understand the latest AI developments.
"""

summary_response = summarizer_chain.invoke({"raw_schedule": schedule_text})
print("--- Executive Summary ---")
print(summary_response.content)

c:\Users\ToniF\miniconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- Executive Summary ---
[{'type': 'text', 'text': '**Schedule Summary:**\n\n* **8:00 AM:** Meeting with the product team regarding the Q3 roadmap.\n* **9:00 AM – 12:00 PM:** Work on LangChain data pipelines.\n* **12:00 PM:** Customer lunch at an Italian restaurant to discuss the latest AI developments.', 'extras': {'signature': 'EoQMCoEMAWkUfRPMy7o1lujVOshWT0vm9SvF7LIMD552l4QsG/YxoYJg9ssKVXLSJN7Qy3B0HL2wirVA+eus1Bh0Mv2mzjqxtSycDb9OjO8GJ8ea7eMqz8SEaaFrtDSl3EB16EDARtn4fkb5lDJT6efySLNnpGkTOGzfgzWsnDSQWY3umUUaRDtYvL6KwTxky0uH9/XP6DQ+yCNHcoeawfdYhRSabCRrnBWdzVOgaVnnX1g6hv0UHrwOd5IgD7Ss+Py5K6kYHY1n9gqzs5vLbwCO8nlMH8qdnn0p4I3vJK4ZqQ1b4DvOIPCl3t1lyJua2WYJ7LsoMVdPgIa3WyFKIGWj1Y8PGhUa5Wg57ivdHHZcty30T0+n5vgbq1NMk1WkthhtB1sErnebOy5Ov2AEsghDukuKVR2VihSyV7oBHXNhPI5ELO3eQBLj3idrEKomI46jSCCUdrrvHXBw/Ci36C/0U63KQCorl2V1yRfDmqFkfNErZWRDTRITpYeOmMGe6Qo3YVyXh9t46+5mZjG5AsTJTarkZRr0WxO9tQ9KJn31RLFXAytHduJQGoeHjGvTAGjdMb8gct98N/6A2uIf7UuQUI7tUAXEj3NDLWFKA9+djrAL65N8coVrvOROuGeMPE87unSPQCujzWd3WIGKM7BMQBF